In [ ]:
from ccka.models.kernel import KernelModel, HardwareKernelRunner, HardwareKernelModel
from ccka.circuits.angleEmbeddingKernel import QuackEmbeddingQiskitCircuit
import pennylane as qml
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
import os
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def _make_circular_data(num_sectors):
    """Generate datapoints arranged in an even circle."""
    center_indices = np.array(range(0, num_sectors))
    sector_angle = 2 * np.pi / num_sectors
    angles = (center_indices + 0.5) * sector_angle
    x = 0.7 * np.cos(angles)
    y = 0.7 * np.sin(angles)
    labels = 2 * np.remainder(np.floor_divide(angles, sector_angle), 2) - 1

    return x, y, labels


def make_double_cake_data(num_sectors):
    x1, y1, labels1 = _make_circular_data(num_sectors)
    x2, y2, labels2 = _make_circular_data(num_sectors)

    # x and y coordinates of the datapoints
    x = np.hstack([x1, 0.5 * x2])
    y = np.hstack([y1, 0.5 * y2])

    # Canonical form of dataset
    X = np.vstack([x, y]).T

    labels = np.hstack([labels1, -1 * labels2])

    # Canonical form of labels
    Y = labels.astype(int)

    return X, Y

X, y = make_double_cake_data(num_sectors=8)
X.shape, y.shape

In [ ]:
"""
Hardware-compatible CentroidBasedKTA using SPSA optimization.

One SPSA step per phase per epoch — exactly matching the original
CentroidBasedKTA.align() structure:
  epoch:
    (1) KAO: one SPSA step jointly on [weights ‖ sub_centroids]
    (2) CO:  one SPSA step on main_centroids only
"""

from __future__ import annotations

import time
from typing import Any

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.svm import SVC


# ─────────────────────────────────────────────────────────────────────────────
# SPSA optimizer
# ─────────────────────────────────────────────────────────────────────────────

class SPSAOptimizer:
    """
    Stateful SPSA optimizer for flat numpy arrays.

        g_k = [f(θ + c_k·Δ) − f(θ − c_k·Δ)] / (2·c_k) · Δ⁻¹
        a_k = a / (A + k + 1)^alpha
        c_k = c / (k + 1)^gamma

    Two circuit evaluations per step regardless of parameter dimension.
    """

    def __init__(
        self,
        a: float     = 0.1,
        c: float     = 0.1,
        A: float     = 10.0,
        alpha: float = 0.602,
        gamma: float = 0.101,
    ) -> None:
        self.a     = a
        self.c     = c
        self.A     = A
        self.alpha = alpha
        self.gamma = gamma
        self._k    = 0

    def step(self, loss_fn, params: np.ndarray) -> np.ndarray:
        """One SPSA update — returns new params."""
        k   = self._k
        a_k = self.a / (self.A + k + 1) ** self.alpha
        c_k = self.c / (k + 1) ** self.gamma

        delta      = np.where(np.random.randint(0, 2, size=params.shape), 1.0, -1.0)
        loss_plus  = loss_fn(params + c_k * delta)
        loss_minus = loss_fn(params - c_k * delta)
        grad       = (loss_plus - loss_minus) / (2.0 * c_k * delta)

        self._k += 1
        return params - a_k * grad


# ─────────────────────────────────────────────────────────────────────────────
# Hardware CentroidBasedKTA
# ─────────────────────────────────────────────────────────────────────────────

class HardwareCentroidKTA:
    """
    Hardware-compatible CentroidBasedKTA with SPSA.

    Epoch structure exactly matches the original:
      (1) ONE KAO step: SPSA jointly on [weights ‖ sub_centroids]
      (2) ONE CO step:  SPSA on main_centroids only

    Parameters
    ----------
    kernel_runner : HardwareKernelRunner
    data          : np.ndarray, shape (N, D)
    labels        : np.ndarray, shape (N,)
    centroids     : int    — sub-centroids per class
    clustering    : {'regular', 'kmeans'}
    lambda_co     : float  — box-constraint penalty weight (CO loss)
    lambda_kao    : float  — L2 regularisation weight (KAO loss)
    split_size    : float  — train fraction
    epochs        : int
    spsa_kao_a    : float  — SPSA step-size scale for KAO
    spsa_co_a     : float  — SPSA step-size scale for CO
    spsa_c        : float  — SPSA perturbation scale (shared)
    spsa_A        : float  — SPSA stability constant (shared)
    """

    def __init__(
        self,
        kernel_runner,
        data: np.ndarray,
        labels: np.ndarray,
        *,
        centroids: int      = 4,
        clustering: str     = "regular",
        lambda_co: float    = 0.001,
        lambda_kao: float   = 0.001,
        split_size: float   = 0.8,
        epochs: int         = 100,
        spsa_kao_a: float   = 0.1,
        spsa_co_a: float    = 0.1,
        spsa_c: float       = 0.1,
        spsa_A: float       = 10.0,
    ) -> None:
        self.kernel_runner = kernel_runner
        self.n_centroids   = centroids
        self.use_kmeans    = clustering.lower() == "kmeans"
        self.lambda_co     = lambda_co
        self.lambda_kao    = lambda_kao
        self.epochs        = epochs

        # ── Data split ────────────────────────────────────────────────────
        self.xtrain, self.xtest, self.ytrain, self.ytest = self._split_data(
            np.array(data,   dtype=np.float64),
            np.array(labels, dtype=np.float64),
            split_size=split_size,
        )

        # ── Weights — flat numpy array ─────────────────────────────────────
        self.weights      = np.array(
            kernel_runner.circuit.init_weights(), dtype=np.float64
        ).ravel()
        self._w_size      = self.weights.size

        # ── Centroids ─────────────────────────────────────────────────────
        (
            self.main_centroids,
            self.main_centroid_labels,
            self.sub_centroids,
            self.sub_centroid_labels,
        ) = self._compute_centroids(self.xtrain, self.ytrain)

        self._sub_shape  = self.sub_centroids.shape   # (n_cls*n_centroids, D)
        self._main_shape = self.main_centroids.shape  # (n_cls, D)

        # ── SPSA optimizers — one per phase, matching original's 3 optimizers
        #   _spsa_kao  ↔  _kao_weight_optimizer + _kao_sub_optimizer (joint)
        #   _spsa_co   ↔  _co_main_optimizer
        self._spsa_kao = SPSAOptimizer(a=spsa_kao_a, c=spsa_c, A=spsa_A)
        self._spsa_co  = SPSAOptimizer(a=spsa_co_a,  c=spsa_c, A=spsa_A)

    # ─────────────────────────────────────────────────────────────────────
    # Data split
    # ─────────────────────────────────────────────────────────────────────

    @staticmethod
    def _split_data(data, labels, split_size=0.8, seed=42):
        rng   = np.random.default_rng(seed)
        perm  = rng.permutation(len(data))
        split = int(len(data) * split_size)
        tr, te = perm[:split], perm[split:]
        return data[tr], data[te], labels[tr], labels[te]

    # ─────────────────────────────────────────────────────────────────────
    # Centroid initialisation — mirrors _compute_centroids exactly
    # ─────────────────────────────────────────────────────────────────────

    def _compute_centroids(self, X, y):
        unique_labels = np.unique(y)
        n_cls, D      = len(unique_labels), X.shape[1]

        main_cents  = np.zeros((n_cls, D),                    dtype=np.float64)
        main_labels = np.zeros((n_cls,),                      dtype=np.float64)
        sub_cents   = np.zeros((n_cls * self.n_centroids, D), dtype=np.float64)
        sub_labels  = np.zeros((n_cls * self.n_centroids,),   dtype=np.float64)

        for ci, label in enumerate(unique_labels):
            class_data      = X[y == label]
            main_cents[ci]  = class_data.mean(axis=0)
            main_labels[ci] = label

            if self.use_kmeans and len(class_data) >= self.n_centroids:
                km = KMeans(
                    n_clusters=self.n_centroids, n_init="auto", random_state=42
                ).fit(class_data)
                sc = km.cluster_centers_
            else:
                chunks = np.array_split(class_data, self.n_centroids)
                sc     = np.stack([c.mean(axis=0) for c in chunks])

            for si in range(self.n_centroids):
                idx             = ci * self.n_centroids + si
                sub_cents[idx]  = sc[si]
                sub_labels[idx] = label

        return main_cents, main_labels, sub_cents, sub_labels

    # ─────────────────────────────────────────────────────────────────────
    # Kernel helpers
    # ─────────────────────────────────────────────────────────────────────

    def _centroid_kernel_vec(self, weights_flat, main_centroid, X):
        """K[i] = kernel(main_centroid, X[i]) for all i — shape (N,)."""
        N  = len(X)
        x0 = np.tile(main_centroid[None, :], (N, 1))
        return self.kernel_runner.forward(x0, X, weights_flat)

    def _full_kernel_matrix(self, weights_flat, X):
        """Symmetric N×N kernel matrix via upper triangle."""
        N       = len(X)
        iu      = np.triu_indices(N)
        k_vals  = self.kernel_runner.forward(X[iu[0]], X[iu[1]], weights_flat)
        K       = np.zeros((N, N), dtype=np.float64)
        K[iu[0], iu[1]] = k_vals
        K[iu[1], iu[0]] = k_vals
        return K

    def _test_kernel_matrix(self, weights_flat, X_train, X_test):
        """M×N cross-kernel: X_test rows × X_train cols."""
        N, M = len(X_train), len(X_test)
        x1   = np.repeat(X_test,  N, axis=0)
        x2   = np.tile(X_train, (M, 1))
        return self.kernel_runner.forward(x1, x2, weights_flat).reshape(M, N)

    # ─────────────────────────────────────────────────────────────────────
    # KTA helpers
    # ─────────────────────────────────────────────────────────────────────

    def _centroid_kta(self, K_vec, y_raw, l):
        """TA(K, Y, l) = l·<K,Y> / (‖K‖·‖Y‖)."""
        Y   = y_raw.astype(np.float64)
        num = l * np.dot(K_vec, Y)
        den = np.linalg.norm(K_vec) * np.linalg.norm(Y) + 1e-10
        return float(num / den)

    def _alignment(self, weights_flat, X, y):
        """Full KTA on the training kernel matrix — for logging only."""
        K   = self._full_kernel_matrix(weights_flat, X)
        T   = np.outer(y, y)
        num = np.sum(K * T)
        den = np.linalg.norm(K, 'fro') * np.linalg.norm(T, 'fro') + 1e-10
        return float(num / den)

    # ─────────────────────────────────────────────────────────────────────
    # Loss functions (scalar, called by SPSA with perturbed params)
    # ─────────────────────────────────────────────────────────────────────

    def _kao_loss(self, joint_flat, main_centroid, y_raw, l):
        """
        KAO loss over joint [weights ‖ sub_centroids] vector.
        Mirrors _loss_kao_cl: 1 − TA(K, Y_raw, l) + λ_kao·L2(weights)
        """
        w   = joint_flat[:self._w_size]
        sc  = joint_flat[self._w_size:].reshape(self._sub_shape)
        K   = self._centroid_kernel_vec(w, main_centroid, sc)
        kta = self._centroid_kta(K, y_raw, l)
        l2  = float(np.sum(w ** 2)) / max(w.size, 1)
        return 1.0 - kta + self.lambda_kao * l2

    def _co_loss(self, main_flat, cl_idx, y_raw, l):
        """
        CO loss over main_centroids flat vector.
        Mirrors _loss_co_cl: 1 − TA(K, Y_raw, l) + λ_co·box_penalty
        sub_centroids are constants here — exactly as in the original.
        """
        main_cents    = main_flat.reshape(self._main_shape)
        main_centroid = main_cents[cl_idx]
        K             = self._centroid_kernel_vec(
                            self.weights, main_centroid, self.sub_centroids
                        )
        kta     = self._centroid_kta(K, y_raw, l)
        penalty = float(np.sum(
            np.maximum(main_centroid - 1.0, 0.0) +
            np.maximum(-main_centroid,       0.0)
        ))
        return 1.0 - kta + self.lambda_co * penalty

    # ─────────────────────────────────────────────────────────────────────
    # SVM evaluation
    # ─────────────────────────────────────────────────────────────────────

    def _svm_eval(self):
        K_train = self._full_kernel_matrix(self.weights, self.xtrain)
        K_test  = self._test_kernel_matrix(self.weights, self.xtrain, self.xtest)

        svm = SVC(kernel="precomputed", C=1.0, probability=True, max_iter=10_000)
        svm.fit(K_train, self.ytrain)

        return {
            "train_accuracy":  float(accuracy_score(self.ytrain, svm.predict(K_train))),
            "test_accuracy":   float(accuracy_score(self.ytest,  svm.predict(K_test))),
            "f1_score":        float(f1_score(       self.ytest,  svm.predict(K_test), average="macro")),
            "precision_score": float(precision_score(self.ytest,  svm.predict(K_test), average="macro")),
            "recall_score":    float(recall_score(   self.ytest,  svm.predict(K_test), average="macro")),
        }

    # ─────────────────────────────────────────────────────────────────────
    # Main training loop
    # ─────────────────────────────────────────────────────────────────────

    def align(self) -> dict[str, Any]:
        unique_labels = np.unique(self.ytrain)
        n_cls         = len(unique_labels)
        y_raw         = self.sub_centroid_labels

        alignment_hist: list[float]      = []
        main_cent_hist: list[np.ndarray] = []
        sub_cent_hist:  list[np.ndarray] = []

        best_test_acc       = -np.inf
        best_weights        = self.weights.copy()
        best_main_centroids = self.main_centroids.copy()
        best_sub_centroids  = self.sub_centroids.copy()

        start = time.perf_counter()

        for epoch in range(self.epochs):

            print(f"Epoch {epoch+1}/{self.epochs} — Circuit executions so far: {self.kernel_runner.circuit_executions}")

            # ── (1) Class selection ────────────────────────────────────────────
            cl       = unique_labels[epoch % n_cls]
            l_kao    = float(cl)
            l_co     = -float(cl)
            main_idx = int(np.argmax(self.main_centroid_labels == cl))

            # ── (2) ONE KAO step: joint [weights ‖ sub_centroids] ─────────────
            joint_flat    = np.concatenate(
                [self.weights.ravel(), self.sub_centroids.ravel()]
            )
            main_centroid = self.main_centroids[main_idx].copy()

            joint_flat = self._spsa_kao.step(
                lambda p: self._kao_loss(p, main_centroid, y_raw, l_kao),
                joint_flat,
            )

            self.weights       = joint_flat[:self._w_size].copy()
            self.sub_centroids = joint_flat[self._w_size:].reshape(self._sub_shape).copy()

            # ── (3) ONE CO step: main_centroids only ──────────────────────────
            main_flat = self.main_centroids.ravel().copy()

            main_flat = self._spsa_co.step(
                lambda p: self._co_loss(p, main_idx, y_raw, l_co),
                main_flat,
            )

            self.main_centroids = np.clip(
                main_flat.reshape(self._main_shape), 0.0, 1.0
            ).copy()

            # ── (4) Log alignment only (no circuit overhead beyond KAO/CO) ─────
           
            main_cent_hist.append(self.main_centroids.copy())
            sub_cent_hist.append(self.sub_centroids.copy())
            print(self.weights)

        # ── Single SVM evaluation at the end ──────────────────────────────────
        final_result = self._svm_eval()
        print(final_result)

        return {
            "weights":                 self.weights,
            "main_centroids":          main_cent_hist,
            "sub_centroids":           sub_cent_hist,
            "final_svm_metrics":       final_result,
            "time":                    time.perf_counter() - start,
            "circuit_executions":      self.kernel_runner.circuit_executions,
        }

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# First time only — saves credentials to disk
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token= '00uaBI1jEfSlimvWilIQ4Z0ERCStsB2gJq2iJla3ZLnz', #'TzFqvDVsCLogQCHMC8uMFVDqoXVesYpXef9i-48OCade',
    overwrite=True,
)

# Load saved account
service = QiskitRuntimeService()

# List available backends and pick the least busy with enough qubits
backend = service.least_busy(
   operational=True,
    simulator=False,
    min_num_qubits=2,          # match your num_qubits
)

print(f"Using backend: {backend.name}")

# ── Circuit and runner ─────────────────────────────────────────────────
num_qubits = 2
reps       = 3

circuit = QuackEmbeddingQiskitCircuit(num_qubits=num_qubits, reps=reps, reupload=True)
runner  = HardwareKernelRunner(
    circuit=circuit,
    backend=backend,
    shots=1024,
    batch_size=1,          # tune to your backend's limits
    optimization_level=3,
    mitigation_level=0,
)

kernel_model = HardwareKernelModel(circuit=circuit, runner=runner)

In [ ]:

kta = HardwareCentroidKTA(
    kernel_runner = kernel_model,
    data          = X,
    labels        = y,

    # centroid settings
    centroids  = 2,           # sub-centroids per class
    clustering = "regular",    # or "regular" for equal-chunk means

    # loss regularisation
    lambda_kao = 0.001,
    lambda_co  = 0.001,

    # training
    split_size = 0.6,
    epochs     = 2,

    # SPSA — KAO phase (kernel weights + sub-centroids)
    spsa_kao_a = 1.0,         # step-size scale; increase if learning too slow
    spsa_co_a  = 1.0,         # step-size scale for centroid updates

    # SPSA — shared perturbation settings
    spsa_c     = 0.1,         # perturbation scale; raise to ~0.3 on noisy hardware
    spsa_A     = 10.0,        # stability constant; raise if early steps are unstable
)

# ── 5. Run alignment ──────────────────────────────────────────────────────────

history = kta.align()

print(history)